## GA Parameter Tuning

The point of this notebook is to tune the parameters of the genetic algorithm. Only mutation probability and warm start ratio are tuned here for time efficiency. Since this is a study and not a productionized pipeline, a faster more brief tuning is enough. Naturally there is also an attempt to increase time complexity to see if it helps. The metric is `'central_tail_effectiveness'` due to the fact that expected shortfall is expected to be the most difficult to optimize and the central variant is less prone to invalid values. Tuning is performed on the second half of validation data (2019).

## Setup

In [1]:
import itertools
import os
from pathlib import Path

import pandas as pd
from joblib import Parallel, delayed

In [2]:
# Find project root (folder that contains .git)
ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

# Set working directory to root
os.chdir(ROOT)

print("Now working in:", Path.cwd())

Now working in: C:\Users\couch\OneDrive\Assignments\Master Thesis\Repo


In [3]:
from src.config import TEST_MODELS

In [4]:
MUTATION_PROBABILITIES = [0.05, 0.2]
WARM_START_RATIOS = [0.2, 0.5]

NGENS = [40, 80]
POP_SIZES = [100, 200]

## Tuning Mutation and Warm Start

In [5]:
def run_single_combination(params):
    import os
    import sys
    from pathlib import Path

    import pandas as pd

    # Find project root (folder that contains .git)
    ROOT = Path.cwd()
    while not (ROOT / ".git").exists():
        ROOT = ROOT.parent

    # Set working directory AND update Python's import path
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

    # Suppress all tqdm progress bars in this process
    os.environ["TQDM_DISABLE"] = "1"

    from src.config import TUNE_END, TUNE_START, WINDOW_SIZE
    from src.simulation import run_backtest

    mutpb, warm_start_ratio, model_tuple = params
    mean_model, volatility_model, _ = model_tuple

    results_dir = Path("results/ga_tuning/mutation_warm_start")

    m_prob_str = str(mutpb).replace(".", "p")
    ws_ratio_str = str(warm_start_ratio).replace(".", "p")

    combo_name = f"mprob_{m_prob_str}_wsr_{ws_ratio_str}"
    combo_dir = results_dir / combo_name
    data_dir = combo_dir / "data"
    logs_dir = combo_dir / "logs"

    data_dir.mkdir(parents=True, exist_ok=True)
    logs_dir.mkdir(parents=True, exist_ok=True)

    result_file = data_dir / f"{mean_model}_{volatility_model}.parquet"
    log_file = logs_dir / f"{mean_model}_{volatility_model}.txt"

    if result_file.exists():
        return

    df = pd.read_parquet("results/processed_data/stock_data.parquet", engine="pyarrow")
    df.index = pd.to_datetime(df.index)
    df = df[df.index <= TUNE_END]

    run_backtest(
        df,
        window_size=WINDOW_SIZE,
        start_date=TUNE_START,
        end_date=TUNE_END,
        mean_model=mean_model,
        volatility_model=volatility_model,
        optimize_portfolio_flag=True,
        save_path=result_file,
        ga_metric="central_tail_effectiveness",
        rf_rates_path="results/processed_data/risk_free_rate.json",
        log_path=log_file,
        ngen=40,
        pop_size=100,
        mutpb=mutpb,
        warm_start_ratio=warm_start_ratio,
    )

    print(f"Finished ({mean_model}, {volatility_model}) for ({mutpb}, {warm_start_ratio})", flush=True)


param_combinations = list(itertools.product(MUTATION_PROBABILITIES, WARM_START_RATIOS, TEST_MODELS))

Parallel(n_jobs=4, backend="loky", batch_size=1)(delayed(run_single_combination)(p) for p in param_combinations)
print("Finished backtest for all parameters")

Finished backtest for all parameters


In [6]:
results_dir = Path("results/ga_tuning/mutation_warm_start")
records = []

# Iterate over all parquet files matching the runner's path structure
for parquet_file in results_dir.glob("mprob_*/data/*.parquet"):
    # Extract parameter folder (e.g., 'mprob_0p05_wsr_0p2')
    combo_folder = parquet_file.parent.parent.name
    parts = combo_folder.split("_")

    mutpb = float(parts[1].replace("p", "."))
    warm_start_ratio = float(parts[3].replace("p", "."))
    model_name = parquet_file.stem

    df = pd.read_parquet(parquet_file, engine="pyarrow")

    # Check for non-finite flags (False values) in ga_metric_is_finite
    warning_mask = (~df["ga_metric_is_finite"]) & df["best_ga_metric_value"].notna()
    if warning_mask.any():
        count = warning_mask.sum()
        print(f"[WARNING] Non-finite GA metric with non-NaN value detected in {parquet_file} ({count} occurrence(s))")

    # Compute overall metric value for this combination
    metric_val = df["best_ga_metric_value"].mean()

    records.append(
        {
            "mutation_probability": mutpb,
            "warm_start_ratio": warm_start_ratio,
            "Model": model_name,
            "metric_value": metric_val,
        }
    )

df_all = pd.DataFrame(records)

# Pivot models into individual columns
summary_df_mprop_wstart = df_all.pivot(
    index=["mutation_probability", "warm_start_ratio"],
    columns="Model",
    values="metric_value",
)

# Extract model column names and calculate row-wise average
summary_df_mprop_wstart["average"] = summary_df_mprop_wstart.mean(axis=1)

# Format numbers: round to 4 decimal places and replace '.' with ','
for col in summary_df_mprop_wstart.columns:
    summary_df_mprop_wstart[col] = summary_df_mprop_wstart[col].apply(lambda x: f"{x:.6f}".replace(".", ","))

summary_df_mprop_wstart

Model                                 naive_naive var_lasso_ccc var_lasso_dcc  \
mutation_probability warm_start_ratio                                           
0.05                 0.2                 0,052250      0,056212      0,056179   
                     0.5                 0,052218      0,056182      0,056163   
0.20                 0.2                 0,052255      0,056265      0,056210   
                     0.5                 0,052251      0,056249      0,056204   

Model                                 var_lasso_go_garch var_lasso_naive  \
mutation_probability warm_start_ratio                                      
0.05                 0.2                        0,052319        0,050583   
                     0.5                        0,052242        0,050574   
0.20                 0.2                        0,052341        0,050603   
                     0.5                        0,052320        0,050601   

Model                                   average  
mutation_probability warm_start_ratio            
0.05                 0.2               0,053509  
                     0.5               0,053476  
0.20                 0.2               0,053535  
                     0.5               0,053525

## Tuning time complexity

In [7]:
def run_single_combination(params):
    import os
    import sys
    from pathlib import Path

    import pandas as pd

    # Find project root (folder that contains .git)
    ROOT = Path.cwd()
    while not (ROOT / ".git").exists():
        ROOT = ROOT.parent

    # Set working directory AND update Python's import path
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

    # Suppress all tqdm progress bars in this process
    os.environ["TQDM_DISABLE"] = "1"

    from src.config import TUNE_END, TUNE_START, WINDOW_SIZE
    from src.simulation import run_backtest

    ngen, pop_size, model_tuple = params
    mean_model, volatility_model, _ = model_tuple

    results_dir = Path("results/ga_tuning/time_complexity")

    combo_name = f"ngen_{ngen}_popsize_{pop_size}"
    combo_dir = results_dir / combo_name
    data_dir = combo_dir / "data"
    logs_dir = combo_dir / "logs"

    data_dir.mkdir(parents=True, exist_ok=True)
    logs_dir.mkdir(parents=True, exist_ok=True)

    result_file = data_dir / f"{mean_model}_{volatility_model}.parquet"
    log_file = logs_dir / f"{mean_model}_{volatility_model}.txt"

    if result_file.exists():
        return

    df = pd.read_parquet("results/processed_data/stock_data.parquet", engine="pyarrow")
    df.index = pd.to_datetime(df.index)
    df = df[df.index <= TUNE_END]

    run_backtest(
        df,
        window_size=WINDOW_SIZE,
        start_date=TUNE_START,
        end_date=TUNE_END,
        mean_model=mean_model,
        volatility_model=volatility_model,
        optimize_portfolio_flag=True,
        save_path=result_file,
        ga_metric="central_tail_effectiveness",
        rf_rates_path="results/processed_data/risk_free_rate.json",
        log_path=log_file,
        ngen=ngen,
        pop_size=pop_size,
        mutpb=0.2,
        warm_start_ratio=0.2,
    )

    print(f"Finished ({mean_model}, {volatility_model}) for ({ngen}, {pop_size})", flush=True)


param_combinations = list(itertools.product(NGENS, POP_SIZES, TEST_MODELS))

Parallel(n_jobs=4, backend="loky", batch_size=1)(delayed(run_single_combination)(p) for p in param_combinations)
print("Finished backtest for all parameters")

Finished backtest for all parameters


In [8]:
results_dir = Path("results/ga_tuning/time_complexity")
records = []

# Iterate over all parquet files matching the runner's path structure
for parquet_file in results_dir.glob("ngen_*/data/*.parquet"):
    combo_folder = parquet_file.parent.parent.name
    parts = combo_folder.split("_")

    ngen = int(parts[1])
    pop_size = int(parts[3])
    model_name = parquet_file.stem

    df = pd.read_parquet(parquet_file, engine="pyarrow")

    # Check for non-finite flags (False values) in ga_metric_is_finite
    warning_mask = (~df["ga_metric_is_finite"]) & df["best_ga_metric_value"].notna()
    if warning_mask.any():
        count = warning_mask.sum()
        print(f"[WARNING] Non-finite GA metric with non-NaN value detected in {parquet_file} ({count} occurrence(s))")

    # Compute overall metric value for this combination
    metric_val = df["best_ga_metric_value"].mean()

    records.append(
        {
            "ngen": ngen,
            "pop_size": pop_size,
            "Model": model_name,
            "metric_value": metric_val,
        }
    )

df_all = pd.DataFrame(records)

# Pivot models into individual columns
summary_df_time_comp = df_all.pivot(
    index=["ngen", "pop_size"],
    columns="Model",
    values="metric_value",
)

# Extract model column names and calculate row-wise average
summary_df_time_comp["average"] = summary_df_time_comp.mean(axis=1)

# Format numbers: round to 4 decimal places and replace '.' with ','
for col in summary_df_mprop_wstart.columns:
    summary_df_time_comp[col] = summary_df_time_comp[col].apply(lambda x: f"{x:.6f}".replace(".", ","))

summary_df_time_comp

Model         naive_naive var_lasso_ccc var_lasso_dcc var_lasso_go_garch  \
ngen pop_size                                                              
40   100         0,052265      0,056285      0,056213           0,052346   
     200         0,052268      0,056279      0,056235           0,052365   
80   100         0,052274      0,056293      0,056245           0,052372   
     200         0,052276      0,056291      0,056246           0,052375   

Model         var_lasso_naive   average  
ngen pop_size                            
40   100             0,050612  0,053544  
     200             0,050615  0,053552  
80   100             0,050625  0,053562  
     200             0,050628  0,053563